# Module `dynamic_network.py`

Ce notebook explique le simulateur dynamique du reseau. Il fait evoluer les vraies aretes du graphe residuel d'un tour sur l'autre, tout en garantissant que le reseau reste exploitable.

`DynamicNetworkSimulator` est volontairement decouple du generateur : un solveur peut recevoir un `DynamicGraphSnapshot` sans savoir d'ou viennent les donnees.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator, active_edge_costs_from_availability
from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig

## 1. Ce que fait le simulateur

A chaque tour :

1. Les couts des vraies aretes sont re-echantillonnes via `sample_dynamic_edge_cost` (mean reversion + bornes).
2. Les aretes peuvent basculer ON/OFF selon `dynamic_forbid_probability` et `dynamic_restore_probability`.
3. Chaque tentative de passage OFF est validee par `DynamicStateValidator` : si elle casse un invariant, elle est refusee.
4. Une fois les aretes mises a jour, on recalcule la matrice residuelle et la fermeture metrique via Dijkstra.

Le simulateur retourne un **nouveau** `DynamicGraphSnapshot`. Le snapshot precedent n'est pas mute : on peut garder l'historique si besoin.

## 2. Creer un simulateur et initialiser

`initialize_snapshot()` demarre tous les couts dynamiques au niveau du cout statique, avec toutes les aretes ON.

In [ ]:
config = GraphGenerationConfig(node_count=10, seed=42,
                              dynamic_sigma=0.18,
                              dynamic_forbid_probability=0.08,
                              dynamic_restore_probability=0.2)
instance = GraphGenerator(config).generate()

simulator = DynamicNetworkSimulator(instance, seed=7)
snapshot = simulator.initialize_snapshot()

print("Tour           :", snapshot.step)
print("Aretes totales :", len(snapshot.edge_availability))
print("Aretes actives :", snapshot.active_edge_count)
print("Premier cout   :", next(iter(snapshot.edge_costs.items())))
print("Premier chemin :", next(iter(snapshot.completed_paths.items())))

## 3. Logique d'`advance()` en detail

Ordre exact des operations pour chaque arete non FORBIDDEN :

```text
si arete ON :
    tirer 'forbid' ~ Uniform(0, 1)
    si forbid < dynamic_forbid_probability ET can_disable_edge(...) :
        passer OFF
sinon (arete OFF) :
    tirer 'restore' ~ Uniform(0, 1)
    si restore < dynamic_restore_probability :
        passer ON (aucune validation necessaire : ajouter une arete ne casse jamais un invariant)

si arete ON apres les tirages :
    cout = sample_dynamic_edge_cost(arete, previous_cost, sigma, mean_rev, max_mult)
```

Apres la boucle sur toutes les aretes, le simulateur filtre les aretes actives (`active_edge_costs_from_availability`), reconstruit la matrice residuelle et relance Dijkstra pour la matrice complete.

## 4. La validation refuse certaines coupures

Pour observer le filet de securite, on boucle sur plusieurs tours en trackant le nombre d'aretes actives.

In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=7)
snapshot = simulator.initialize_snapshot()

print(f"tour={snapshot.step:>2}  actives={snapshot.active_edge_count:>3}")
for _ in range(20):
    snapshot = simulator.advance(snapshot)
    print(f"tour={snapshot.step:>2}  actives={snapshot.active_edge_count:>3}")

Meme avec `dynamic_forbid_probability` eleve, le nombre d'aretes actives ne descend jamais sous le plancher fixe par :

- la connexite (au moins `n-1` aretes) ;
- `resolved_dynamic_min_density` et `resolved_dynamic_min_average_degree` ;
- `dynamic_max_disabled_ratio`.

## 5. Recalcul Dijkstra a chaque tour

Apres chaque advance, la methode privee `_build_snapshot` :

1. prend les aretes actives ;
2. construit `residual_costs` avec `build_cost_matrix` ;
3. applique `complete_graph_with_shortest_paths` pour obtenir `completed_costs` et `completed_paths`.

Les solveurs ne voient jamais les aretes OFF : elles n'existent tout simplement pas dans la matrice qu'ils consomment via `SolverInput`.

In [ ]:
print("Ligne 0 de la matrice complete :")
print(snapshot.completed_costs[0])
print("\nChemin (0, 3) :", snapshot.completed_paths.get((0, 3)))

## 6. Helper : `active_edge_costs_from_availability`

Fonction libre qui filtre un dict de couts en ne gardant que les aretes marquees ON. Exposee parce qu'elle est utile en debug et hors du simulateur.

In [ ]:
subset = active_edge_costs_from_availability(snapshot.edge_costs, snapshot.edge_availability)
print("Nb total d'aretes connues   :", len(snapshot.edge_costs))
print("Nb d'aretes actives filtrees :", len(subset))

## 7. Invariants finaux

Apres un appel a `advance()`, les proprietes suivantes sont toujours vraies (par construction + validation) :

- le graphe actif est connexe ;
- la matrice `completed_costs` respecte l'inegalite triangulaire ;
- chaque `completed_paths[(i, j)]` utilise uniquement des aretes actives du tour ;
- aucun cout actif ne descend sous `base_cost` ni ne depasse `static_cost * dynamic_max_multiplier`.

Ces garanties permettent aux solveurs metaheuristiques de tourner sans verifier eux-memes la coherence du graphe a chaque tour.